# 10 - Audit and replay terminal work

Requeues one terminal/dead-lettered item after operator approval. A stable `REPLAY_ID` makes the three-step audit/requeue/finalize workflow retry-safe, including interruption after the work row changes but before `applied_at` is recorded.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
REPLAY_ID = ""
WORK_ID = ""
REQUESTED_BY = ""
REASON = ""
MAX_ATTEMPTS = 4
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timezone
import json
import re

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
TERMINAL = {"TERMINAL_FAILED", "DEAD_LETTERED"}
REPLAYED = {"QUEUED", "LEASED", "STAGING", "RUNNING", "WRITING", "RETRY_WAIT", "SUCCEEDED"}


def required(value: str, name: str, limit: int) -> str:
    parsed = value.strip()
    if not parsed or len(parsed) > limit:
        raise ValueError(f"{name} must contain 1-{limit} characters")
    return parsed


replay_id = required(REPLAY_ID, "REPLAY_ID", 200)
work_id = required(WORK_ID, "WORK_ID", 200)
requested_by = required(REQUESTED_BY, "REQUESTED_BY", 200)
reason = required(REASON, "REASON", 2000)
max_attempts = int(MAX_ATTEMPTS)
if max_attempts < 1:
    raise ValueError("MAX_ATTEMPTS must be at least 1")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
work_table = table("video_work")
replay_table = table("replay_requests")


def one_row(table_name: str, column: str, value: str) -> dict:
    rows = spark_session.table(table_name).where(F.col(column) == value).limit(2).collect()
    if len(rows) > 1:
        raise RuntimeError(f"Duplicate {column} rows exist for {value}")
    return rows[0].asDict(recursive=True) if rows else {}


def finalize_request(request: dict, applied_at: datetime) -> None:
    source = spark_session.createDataFrame(
        [(request["replay_id"], request["capture_date"], applied_at)],
        "replay_id string, capture_date date, applied_at timestamp",
    )
    (
        DeltaTable.forName(spark_session, replay_table)
        .alias("t")
        .merge(source.alias("s"), "t.replay_id = s.replay_id AND t.capture_date = s.capture_date")
        .whenMatchedUpdate(set={"applied_at": "s.applied_at"})
        .execute()
    )


request = one_row(replay_table, "replay_id", replay_id)
work = one_row(work_table, "work_id", work_id)
if not work:
    raise RuntimeError(f"Work row does not exist: {work_id}")

if request:
    for field, expected in (("work_id", work_id), ("requested_by", requested_by), ("reason", reason)):
        if request[field] != expected:
            raise ValueError(f"REPLAY_ID already belongs to a different {field}")
    if request["applied_at"] is not None:
        outcome = {**request, "status": work["status"]}
    elif work["status"] in REPLAYED:
        applied_at = datetime.now(timezone.utc)
        finalize_request(request, applied_at)
        outcome = {**request, "applied_at": applied_at, "status": work["status"]}
    elif work["status"] not in TERMINAL:
        raise ValueError(f"Existing replay cannot be reconciled with status={work['status']}")

if not request or (request["applied_at"] is None and work["status"] in TERMINAL):
    if work["committed_attempt_id"] is not None or work["status"] == "SUCCEEDED":
        raise ValueError("Committed work cannot be replayed")
    if work["status"] not in TERMINAL:
        raise ValueError(f"Work is not eligible for replay: status={work['status']}")
    requested_at = request.get("requested_at") or datetime.now(timezone.utc)
    request_row = request or {
        "replay_id": replay_id,
        "work_id": work_id,
        "requested_by": requested_by,
        "reason": reason,
        "requested_at": requested_at,
        "previous_status": work["status"],
        "applied_at": None,
        "capture_date": work["capture_date"],
    }
    request_source = spark_session.createDataFrame([request_row], spark_session.table(replay_table).schema)
    (
        DeltaTable.forName(spark_session, replay_table)
        .alias("t")
        .merge(request_source.alias("s"), "t.replay_id = s.replay_id AND t.capture_date = s.capture_date")
        .whenNotMatchedInsertAll()
        .execute()
    )
    work_source = spark_session.createDataFrame(
        [(work_id, work["capture_date"], max_attempts)],
        "work_id string, capture_date date, max_attempts int",
    )
    (
        DeltaTable.forName(spark_session, work_table)
        .alias("t")
        .merge(work_source.alias("s"), "t.work_id = s.work_id AND t.capture_date = s.capture_date")
        .whenMatchedUpdate(
            condition=(
                "t.status IN ('TERMINAL_FAILED', 'DEAD_LETTERED') AND "
                "t.committed_attempt_id IS NULL"
            ),
            set={
                "status": "'QUEUED'", "queued_at": "current_timestamp()",
                "not_before_at": "NULL", "attempt_count": "0",
                "max_attempts": "s.max_attempts", "lease_owner_attempt_id": "NULL",
                "lease_dispatcher_id": "NULL", "lease_acquired_at": "NULL",
                "lease_expires_at": "NULL", "last_heartbeat_at": "NULL",
                "last_error_category": "NULL", "last_error_type": "NULL",
                "last_error_message": "NULL",
            },
        )
        .execute()
    )
    updated = one_row(work_table, "work_id", work_id)
    if updated["status"] not in REPLAYED:
        raise RuntimeError("Replay request was recorded but work was not requeued")
    applied_at = datetime.now(timezone.utc)
    finalize_request(request_row, applied_at)
    outcome = {**request_row, "applied_at": applied_at, "status": updated["status"]}

print(json.dumps(outcome, default=str, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, default=str, sort_keys=True))